# 05 — Macro Analysis

Analyse funding rates, open interest, and VIX proxy signals.

In [ ]:
import sys
sys.path.insert(0, '../../')

import pandas as pd
import numpy as np
from research.utils.data_loader import load_candles, load_funding_rates, load_open_interest
from research.utils.feature_engineering import add_all_indicators, add_funding_features
from research.utils.visualization import funding_rate_chart, indicator_chart, candlestick_chart

SYMBOL = 'BTC/USDT:USDT'

In [ ]:
funding_df = load_funding_rates(symbol=SYMBOL)
print(f'Funding rate records: {len(funding_df)}')
funding_df.head()

In [ ]:
fig = funding_rate_chart(funding_df, symbol=SYMBOL)
fig.show()

In [ ]:
oi_df = load_open_interest(symbol=SYMBOL)
print(f'Open Interest records: {len(oi_df)}')

if not oi_df.empty:
    fig = indicator_chart(oi_df, ['oi_usd'], title=f'Open Interest — {SYMBOL}')
    fig.show()

In [ ]:
df = load_candles('binance', SYMBOL, '4h', limit=5000)
df = add_all_indicators(df)
df = add_funding_features(df, funding_df)

if 'funding_rate' in df.columns:
    corr = df[['close', 'funding_rate', 'funding_zscore', 'rsi_14', 'rvol_20']].corr() if 'rvol_20' in df.columns else df[['close', 'funding_rate']].corr()
    print('Correlation matrix:')
    print(corr)

In [ ]:
if 'funding_zscore' in df.columns:
    extreme_mask = df['funding_zscore'].abs() > 2
    extreme_events = df[extreme_mask]
    if not extreme_events.empty:
        fwd_ret = df['close'].pct_change(8).shift(-8)
        extreme_events = extreme_events.copy()
        extreme_events['fwd_ret_8h'] = fwd_ret[extreme_mask]
        print(f'Extreme funding events (|z| > 2): {len(extreme_events)}')
        print(extreme_events[['funding_rate', 'funding_zscore', 'fwd_ret_8h']].describe())